# Notebook 03: Add a New Problem Scaffold (DCC26)

Reference implementation of a minimal, reproducible benchmark problem scaffold.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

Use this as a pattern for structuring new benchmark problems with explicit contracts and validation checks.


## What makes a new problem benchmark-ready

Benchmark value comes from clarity and comparability, not only simulator sophistication.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'matplotlib', 'gymnasium']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


### Step 1 - Import scaffold dependencies

Keep imports minimal and interface-focused.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem


### Step 2 - Implement minimal problem contract

Ensure methods are deterministic and constraints/objectives are semantically explicit.


In [ ]:
class ToyDensityProblem(Problem[np.ndarray]):
    """Minimal toy problem scaffold for workshop teaching."""

    version = 0
    objectives = (("toy_cost", ObjectiveDirection.MINIMIZE),)

    @dataclass
    class Conditions:
        target_density: Annotated[float, bounded(lower=0.0, upper=1.0)] = 0.5

    @dataclass
    class Config(Conditions):
        resolution: Annotated[int, bounded(lower=4, upper=128)] = 16
        max_iter: Annotated[int, bounded(lower=1, upper=200)] = 20

    dataset_id = "IDEALLab/beams_2d_50_100_v0"  # placeholder to satisfy scaffold
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(target_density=self.config.target_density)
        self.design_space = spaces.Box(
            low=0.0, high=1.0, shape=(self.config.resolution, self.config.resolution), dtype=np.float32
        )

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        cfg = {"target_density": self.config.target_density, **(config or {})}
        # Toy objective: mismatch to target density + smoothness penalty
        density_term = abs(float(design.mean()) - float(cfg["target_density"]))
        smoothness = float(np.mean(np.abs(np.diff(design, axis=0))))
        return np.array([density_term + 0.1 * smoothness], dtype=np.float32)

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        cfg = {"target_density": self.config.target_density, **(config or {})}
        x = starting_point.copy().astype(np.float32)
        hist = []
        for step in range(self.config.max_iter):
            # Toy update toward target density (not a real optimizer)
            x = np.clip(x + 0.2 * (cfg["target_density"] - x), 0.0, 1.0)
            hist.append(OptiStep(obj_values=self.simulate(x, cfg), step=step))
        return x, hist

    def render(self, design: np.ndarray, *, open_window: bool = False):
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(design, cmap="viridis", vmin=0, vmax=1)
        ax.set_title("ToyDensityProblem design")
        ax.axis("off")
        if open_window:
            plt.show()
        return fig, ax

    def random_design(self):
        d = self.np_random.random(self.design_space.shape).astype(np.float32)
        return d, -1


### Step 3 - Smoke-test the scaffold

Validate behavior with simple checks before scaling to real domains.


In [ ]:
problem = ToyDensityProblem(seed=42, resolution=16, target_density=0.4, max_iter=10)
start, _ = problem.random_design()

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config={'target_density': 0.4, 'resolution': 16, 'max_iter': 10})
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config={'target_density': 0.4})
opt_design, history = problem.optimize(start, config={'target_density': 0.4})
objf = problem.simulate(opt_design, config={'target_density': 0.4})

print('initial objective:', float(obj0[0]))
print('final objective:', float(objf[0]))
print('optimization steps:', len(history))

problem.render(opt_design)

## Mapping to real EngiBench contributions

Use this template to onboard new domains while preserving common evaluation semantics.


## Contribution checklist

Check for leakage risks, undocumented defaults, and missing reproducibility metadata before contribution.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
